In [1]:
from models_HyperNetwork import HyperNetworkSEModel
from models_LoRA import LoRASEModel
from generate_utils import load_GraphModel, load_BiLSTMModel, generate_files_with_nucleus
import torch
import numpy as np
import pickle
from tqdm import tqdm
from GridMLM_tokenizers import CSGridMLMTokenizer
from graph_utils import get_graph_embeddings_from_string_with_model, get_bilstm_embeddings_from_string_with_model

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
device_name = 'cuda:0'
device = torch.device(device_name)

graph_model_path = 'saved_models/HyperNetwork/graph/graph_model.pt'
transformer_graph_path = 'saved_models/HyperNetwork/graph/transformer_model.pt'

bilstm_model_path = 'saved_models/HyperNetwork/bilstm/bilstm_model.pt'
transformer_bilstm_path = 'saved_models/HyperNetwork/bilstm/transformer_model.pt'

In [4]:
graph_model = load_GraphModel(graph_model_path, device)
bilstm_model = load_BiLSTMModel(bilstm_model_path, device)

graph_model.eval()
bilstm_model.eval()

RuntimeError: Error(s) in loading state_dict for HarmonyBiLSTM:
	size mismatch for input_proj.0.weight: copying a param with shape torch.Size([64, 12]) from checkpoint, the shape in current model is torch.Size([256, 12]).
	size mismatch for input_proj.0.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for lstm.weight_ih_l0: copying a param with shape torch.Size([512, 64]) from checkpoint, the shape in current model is torch.Size([1024, 256]).
	size mismatch for lstm.weight_hh_l0: copying a param with shape torch.Size([512, 128]) from checkpoint, the shape in current model is torch.Size([1024, 256]).
	size mismatch for lstm.bias_ih_l0: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([1024]).
	size mismatch for lstm.bias_hh_l0: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([1024]).
	size mismatch 

In [ ]:
datasets = {
    'gjt': {'path': 'data/gjt_test.pkl', 'dataset': None},
    # 'hook': {'path': 'data/hook_test.pkl', 'dataset': None},
    'nott': {'path': 'data/nott_test.pkl', 'dataset': None},
    # 'wiki': {'path': 'data/wiki_test.pkl', 'dataset': None}
}

for k, v in datasets.items():
    print(f'loading {k}')
    with open(v['path'], 'rb') as f:
        d = pickle.load(f)
    v['dataset'] = d

loading gjt
loading nott


In [ ]:
in_seq = 'E:min;A:maj;F:min;A#:maj'

y_graph = get_graph_embeddings_from_string_with_model(in_seq, graph_model)
y_bilstm = get_bilstm_embeddings_from_string_with_model(in_seq, bilstm_model)

E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297
E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297


In [4]:
transformer_model = HyperNetworkSEModel(
    len(tokenizer.vocab),
    512, #graph_model.output_dim,
    device,
    d_model=512,
    nhead=8,
    num_layers=8,
    dim_feedforward=2048,
    pianoroll_dim=13,
    grid_length=80,
    dropout=0.3
)

transformer_model.to(device)

HyperNetworkSEModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (hyper_q): HyperGuide(
    (layer_embedding): Embedding(8, 16)
    (head_embedding): Embedding(8, 16)
    (mlp): Sequential(
      (0): Linear(in_features=544, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=128, bias=True)
      (3): GELU(approximate='none')
    )
    (gamma_proj): Linear(in_features=128, out_features=64, bias=True)
    (beta_proj): Linear(in_features=128, out_features=64, bias=True)
    (lora_A): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (lora_B): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
  )

In [5]:
lora_model = LoRASEModel(
    len(tokenizer.vocab),
    512, #graph_model.output_dim,
    device,
    d_model=512,
    nhead=8,
    num_layers=8,
    dim_feedforward=2048,
    pianoroll_dim=13,
    grid_length=80,
    dropout=0.3
)

lora_model.to(device)

LoRASEModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (layers): ModuleList(
    (0-7): 8 x TransformerBlockWithAttnLoRA(
      (attn): MultiHeadAttentionWithAttnLoRA(
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
        (q_lora): ModuleList(
          (0-7): 8 x HyperLoRA(
            (lora_A): Linear(in_features=512, out_features=32, bias=True)
            (lora_B): Linear(in_features=512, out_features=32, bias=True)
          )
        )
        (k_lora): ModuleList(
          (0-7): 8 x HyperLoRA(
            (lora_A): Linear(in_features=512, out_features=32, bias=True)
            (lora_B): Linear(in_features=512, out_features=32, bias=True)
          )
        )
        (v

In [6]:
def count_parameters(model):
    """Count total and trainable parameters in a model."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Compare the two models
total_hyper, trainable_hyper = count_parameters(transformer_model)
total_lora, trainable_lora = count_parameters(lora_model)

print(f"HyperNetwork Model:")
print(f"  Total params: {total_hyper:,}")
print(f"  Trainable params: {trainable_hyper:,}")
print(f"\nLoRA Model:")
print(f"  Total params: {total_lora:,}")
print(f"  Trainable params: {trainable_lora:,}")
print(f"\nDifference: {abs(total_hyper - total_lora):,} params")

HyperNetwork Model:
  Total params: 29,053,350
  Trainable params: 29,053,350

LoRA Model:
  Total params: 35,435,043
  Trainable params: 35,435,043

Difference: 6,381,693 params


In [8]:
x = datasets['gjt']['dataset'][0]
print(x)

{'harmony_ids': [6, 217, 217, 217, 217, 6, 217, 217, 217, 217, 6, 194, 194, 194, 194, 6, 332, 332, 332, 332, 6, 131, 131, 131, 131, 6, 131, 131, 131, 131, 6, 80, 80, 80, 80, 6, 231, 231, 231, 231, 6, 14, 14, 14, 14, 6, 168, 168, 168, 168, 6, 217, 217, 217, 217, 6, 131, 131, 131, 131, 6, 284, 284, 284, 284, 6, 284, 284, 284, 284, 6, 276, 276, 276, 276, 6, 71, 71, 71, 71], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'pianoroll': array([[0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], dtype=float32), 'time_signature': [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], 'h_density_complexity': [1, 

In [9]:
y_graph = get_graph_embeddings_from_string_with_model(in_seq, graph_model)

E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297


In [10]:
y = transformer_model(
    torch.tensor(x['pianoroll'], dtype=torch.float32).unsqueeze(0).to(device),
    z_g=y_graph.unsqueeze(0)
)